# backprop-pop-outgrad-loop — ex1: implement the main reverse-pass loop over a sorted graph

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `backprop-pop-outgrad-loop`. Running the final beacon cell reports progress against the `Backprop: backprop pop-outgrad loop` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: backprop pop-outgrad loop` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backprop-pop-outgrad-loop`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backprop-pop-outgrad-loop"
DD_SUBTOPIC = "Backprop: backprop pop-outgrad loop"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Backprop pop-outgrad loop — quick refresher

The main reverse-pass driver. Walk the graph in reverse-topological order (end node first), pop each node's accumulated grad out of a `grads` dict, dispatch the back_fn for each parent, accumulate into the parent's slot:

```python
grads = {id(end_node): end_grad}   # seed with dL/d(end_node)
for node in sorted_computational_graph(end_node):
    grad_out = grads.pop(id(node))     # pop — node done after this
    if node.recipe is None:            # leaf: write to .grad
        node.grad = grad_out if node.grad is None else node.grad + grad_out
        continue
    for argnum, parent in node.recipe.parents.items():
        back_fn = BACK_FUNCS.get_back_func(node.recipe.func, argnum)
        grad_parent = back_fn(grad_out, node.array,
                              *node.recipe.args, **node.recipe.kwargs)
        grads[id(parent)] = grads.get(id(parent), 0) + grad_parent
```

Three invariants:
- **Pop, don't peek.** Once we process a node, its grad is no longer   needed; popping frees it and surfaces bugs where a node's grad got   consumed before all parents accumulated.
- **Accumulate with `+`, never overwrite.** Diamond graphs route grad   through multiple paths; the same parent shows up in multiple   `recipe.parents` walks.
- **Leaves get `.grad`, non-leaves stay in `grads` dict.** Leaves are   the user-facing parameters; non-leaves are intermediate.

### Exercise 1 — implement the main reverse-pass loop over a sorted graph

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the reverse-pass driver pattern: iterate a sorted-computational-graph, pop each node's accumulated grad, dispatch the per-arg back_fn, and accumulate into each parent's grad slot.
> Keywords: reverse-pass, loop, pop-grad, accumulate, leaf-grad
> ```

**KCs targeted:** `backprop-pop-outgrad-loop`, `dispatch-back-fn-from-recipe`

Implement `backprop(end_node, end_grad, sorted_graph, back_funcs)` — the main reverse-pass driver.

**Inputs.**
- `end_node` — the MiniTensor at which to start the reverse pass (e.g. the loss).
- `end_grad` — a `torch.Tensor` with `dL/d(end_node)`. Usually `t.ones_like(end_node.array)`.
- `sorted_graph` — list of MiniTensors in REVERSE-topological order (end_node FIRST, leaves LAST). Pre-computed by the caller.
- `back_funcs` — a dict `{(forward_fn, argnum): back_fn}` that maps to the right back_fn for each (op, arg-position) pair.

**Algorithm.**

```python
grads = {id(end_node): end_grad}            # seed accumulator
for node in sorted_graph:
    grad_out = grads.pop(id(node))          # pop — node done after
    if node.recipe is None:                 # leaf: write to .grad
        if node.grad is None:
            node.grad = grad_out
        else:
            node.grad = node.grad + grad_out
        continue
    for argnum, parent in node.recipe.parents.items():
        back_fn = back_funcs[(node.recipe.func, argnum)]
        grad_parent = back_fn(grad_out, node.array,
                              *node.recipe.args,
                              **node.recipe.kwargs)
        pid = id(parent)
        grads[pid] = grads.get(pid, 0) + grad_parent
```

**Three invariants the test checks.**
1. **Pop, don't peek.** `grads.pop` removes the entry; later accidental reads should hit `KeyError`.
2. **Accumulate with `+`, never overwrite.** Diamond DAGs route grad through multiple paths; the same parent appears in multiple `recipe.parents` walks.
3. **Leaves write to `.grad`; non-leaves stay in `grads` dict.** Leaves are the user-facing parameters; non-leaves are intermediate.

**No return value.** Mutate `.grad` on each leaf MiniTensor in-place. The dispatcher is a one-pass walk.

In [ ]:
def backprop(end_node, end_grad, sorted_graph, back_funcs) -> None:
    grads = {id(end_node): end_grad}
    for node in sorted_graph:
        nid = id(node)
        if nid not in grads:
            # Node not reached from end_node in this traversal — skip.
            continue
        grad_out = grads.pop(nid)           # POP — node done after this
        if node.recipe is None:
            # Leaf: accumulate (don't overwrite) into .grad.
            if node.grad is None:
                node.grad = grad_out
            else:
                node.grad = node.grad + grad_out
            continue
        # Non-leaf: dispatch + accumulate into each parent.
        for argnum, parent in node.recipe.parents.items():
            back_fn = back_funcs[(node.recipe.func, argnum)]
            grad_parent = back_fn(
                grad_out, node.array,
                *node.recipe.args, **node.recipe.kwargs,
            )
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + grad_parent


<details><summary>Solution</summary>

```python
def backprop(end_node, end_grad, sorted_graph, back_funcs) -> None:
    grads = {id(end_node): end_grad}
    for node in sorted_graph:
        nid = id(node)
        if nid not in grads:
            # Node not reached from end_node in this traversal — skip.
            continue
        grad_out = grads.pop(nid)           # POP — node done after this
        if node.recipe is None:
            # Leaf: accumulate (don't overwrite) into .grad.
            if node.grad is None:
                node.grad = grad_out
            else:
                node.grad = node.grad + grad_out
            continue
        # Non-leaf: dispatch + accumulate into each parent.
        for argnum, parent in node.recipe.parents.items():
            back_fn = back_funcs[(node.recipe.func, argnum)]
            grad_parent = back_fn(
                grad_out, node.array,
                *node.recipe.args, **node.recipe.kwargs,
            )
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + grad_parent
```

**Why `grads.pop` instead of `grads[nid]`.** Once we've processed a node, its accumulated grad is no longer needed — every parent that wanted to add to it has already done so (that's what the reverse-topological ordering buys us). Popping surfaces bugs where a node's grad is consumed before all incoming grads have been accumulated; a later read would `KeyError`.

**Why `grads.get(pid, 0) + grad_parent` not assignment.** A single parent can appear in multiple `recipe.parents` walks: (a) the diamond DAG case (one node used twice in a forward op), (b) the multi-consumer case (one node feeds multiple downstream ops, each of which will eventually contribute grad). The accumulator pattern handles both.

**Why leaves get `.grad`, non-leaves stay in `grads`.** The user-facing API of an autograd Tensor is `.grad`. But during the reverse pass we need a scratch dict for non-leaf intermediates (we never expose those). Splitting the storage by `node.recipe is None` is the cheap classifier — leaves have no recipe by construction.

**Why pass `*node.recipe.args, **node.recipe.kwargs`.** The back_fn signature is `(grad_out, out, *args, **kwargs)`. The Recipe stored the raw call-time args (unboxed tensors) and kwargs (dim, keepdim, ...) so reverse-pass dispatch is generic. Drop either and shape-mismatched ops will blow up.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()